# Tutorial 1: open your first images

In this tutorial you will:

1. make a small test image,
2. start the viewer and open the image in your browser,
3. try a timelapse, a folder of positions, and a run that is still being written,
4. mark places on the image.

You need the viewer installed and a browser. No microscope.

```bash
pip install "zmart-viewer @ git+https://github.com/thomdehoog/ZMART-viewer"
```

From a terminal you would type `zmart-viewer /path/to/data`. That command blocks the terminal, so this notebook starts the same engine from Python and opens things through the same door. Each step also shows the terminal command that does the same.

## 1. Set up

`sample_data.py` sits next to this notebook. It writes small synthetic OME-Zarr images, so you have something to look at.

In [ ]:
import json
import threading
import time
import urllib.request
from pathlib import Path

import numpy as np
from sample_data import a_specimen, write_image
from zmart_viewer import make_server

work = Path("tutorial-data").resolve()
work.mkdir(exist_ok=True)
work

## 2. Make a test image

`a_specimen()` returns a small volume of blob-like cells with two channels. `write_image()` saves it as an OME-Zarr image: the full picture plus two smaller copies, with a description of the axes, the pixel size and the channels.

In [ ]:
image = write_image(work / "cells.ome.zarr", a_specimen())
sorted(p.name for p in image.iterdir())

One folder per copy of the image (`0` is full size, `1` is half, `2` is a quarter) and one `zarr.json` that describes them. That is all an OME-Zarr image is.

## 3. Start the viewer

Terminal: `zmart-viewer tutorial-data/cells.ome.zarr`

Here we start the engine on a free port and keep a small `show()` function that opens a path in it, replacing what was open. It uses the same request the command and the load window use, so a path opens exactly as it would there.

In [ ]:
server = make_server(port=0, data_dir=work, live=True, allow_selection=True)
threading.Thread(target=server.serve_forever, daemon=True).start()
address = f"http://127.0.0.1:{server.server_address[1]}"


def ask(route, payload=None):
    """One request to the engine: GET without a payload, POST with one."""
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(
        f"{address}{route}", data=data, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(request, timeout=600) as answer:
        return json.loads(answer.read())


def show(path):
    """Open `path` in the viewer, replacing what is open, and return the rows shown."""
    for group in ask("/api/config")["groups"]:
        ask("/api/stores/close", {"group": group})
    config = ask("/api/stores/open", {"path": str(path)})
    return [layer["name"] for layer in config["layers"]]


print("Open this address in your browser:", address)

In [ ]:
show(image)

The viewer now shows the image, one row per channel under **DATA**.

## 4. Look around

| Do | See |
|---|---|
| Drag | The image moves under the pointer |
| Turn the wheel | Zoom, about the pointer |
| Move the **Z** slider on the right, or Shift + wheel | Another plane of the stack |
| Press ▶ next to the slider | The planes play on their own |
| Click **marker** under **DATA** | That channel is selected |
| Look at **CHANNEL SETTINGS** | Its histogram, **MIN** and **MAX** (the black and white points), **OPACITY** and colour |
| Press **Auto** | A display window measured from what is on screen |
| Click the eye next to a channel | The channel is hidden |
| Press **3D** at the top left | The stack as a volume. **2D** brings the planes back |
| Press **Overview** | The view as it opened: straight, centred, sized to the window |

The **‹** handle folds the bar of controls away when you want the whole screen for the image.

## 5. A timelapse

A timelapse is the same image with a time axis in front. Write five frames in which the cells drift a little.

In [ ]:
frames = np.stack([a_specimen(frame=n) for n in range(5)])
timelapse = write_image(work / "timelapse.ome.zarr", frames)
show(timelapse)

A **T** slider has appeared along the bottom, with its own ▶. The viewer shows a slider only when the image really has that axis with more than one step.

Terminal: `zmart-viewer tutorial-data/timelapse.ome.zarr`

## 6. Many positions, one picture

A microscope that surveys a specimen writes one image per stage position. Each image records where on the stage it was taken. Give the viewer the folder, and it draws the positions as one picture, each where it belongs.

Write a two-by-two survey. The positions are 256 pixels wide and 224 micrometres apart, so neighbours overlap a little, as they do on a real stage.

In [ ]:
STEP_UM = 224.0
PLACES = [(0, 0), (0, 1), (1, 0), (1, 1)]  # (row, column) on the stage


def write_position(folder, number, *, seed=None):
    """Write position `number` of a survey into `folder`."""
    row, column = PLACES[number]
    return write_image(
        folder / f"pos{number:02d}.ome.zarr",
        a_specimen(seed=seed if seed is not None else number),
        origin_um=(row * STEP_UM, column * STEP_UM),
    )


survey = work / "survey"
for number in range(4):
    write_position(survey, number)
show(survey)

Still two rows, **stain** and **marker**: one picture, not four. Zoom out to see all four positions in place.

Nothing was copied. The four images are exactly as written. The viewer only links them together for this session. In the load window you can tick **build the low-resolution mosaic and keep the view** to keep that link and a small zoomed-out copy, so a large survey opens instantly next time.

Terminal: `zmart-viewer tutorial-data/survey`

## 7. A run that is still being written

The viewer follows the folder by default. Open a run that holds its first position, then write three more while you watch the browser. Each one appears within a second.

Keep the browser visible while the next cell runs.

In [ ]:
run = work / "run"
write_position(run, 0)
print(show(run))

for number in (1, 2, 3):
    time.sleep(3)
    write_position(run, number)
    print("the microscope wrote position", number)

When a run has finished, start the viewer with `--static`. It then stops looking for new images, which is the largest thing it does on a big folder, and moving through old data feels instant.

Terminal: `zmart-viewer tutorial-data/run` while the run is written, `zmart-viewer tutorial-data/run --static` afterwards.

## 8. Mark places

We started the viewer with `allow_selection=True`, which is `--select` on the command line. That shows the **SELECTION** panel.

1. Under **SELECTION**, choose **Point** or **Box**.
2. Hold Ctrl and click in the image. For a box, Ctrl + click two corners.
3. Give the mark a name.

Marks are saved a moment after you make them. There is no save button. The file sits next to your images:

In [ ]:
marks = work / "zmart-annotations.json"
if marks.exists():
    print(json.dumps(json.loads(marks.read_text()), indent=1))
else:
    print("No marks yet. Draw one in the browser, then run this cell again.")

The viewer never moves a microscope. Acting on a mark is up to the software that runs the experiment, which can read this file.

## 9. Stop the viewer

In [ ]:
server.shutdown()
server.server_close()

## What you learned

| Terminal | Does |
|---|---|
| `zmart-viewer image.ome.zarr` | Opens one image |
| `zmart-viewer folder` | Opens a folder of positions as one picture, and follows it |
| `zmart-viewer folder --static` | The same, for a finished run |
| `zmart-viewer folder --select` | Also shows the **SELECTION** panel |
| `zmart-viewer folder --no-window` | Prints an address to open in a browser |
| `zmart-viewer` | An empty viewer with a **LOAD DATA** button |

Next: [Tutorial 2](02_the_engine_in_your_own_interface.ipynb) puts the picture inside your own interface. The [guide](../README.md) has everything else.